# Sesión 4 · Modelos supervisados: lineales, Naive Bayes, k-NN y SVM
**IA aplicada a la ciberseguridad · Máster en Ciberseguridad**

| Parte | Qué haces |
|---|---|
| A | Regresión lineal sobre logins de una VPN y anomalías por residuo |
| B | Regresión logística desde cero |
| C | Modelo completo sobre NSL-KDD: odds y umbral por coste |
| D | Naive Bayes para texto de phishing |
| E | k-NN y SVM: el efecto del escalado |
| F | Clasificación multiclase de ataques |

> **Dos rutas en este cuaderno.** La **ruta base** ejecuta y observa: cada parte explica qué deberías ver. Los bloques **Reto avanzado** son opcionales y piden método.
>
> Ruta base: partes A a E. Ruta avanzada: retos y parte F.
>
> Ejecuta las celdas **en orden** (`Mayús + Intro`). En Colab, si se desconecta el entorno, vuelve a ejecutar la preparación.

## Preparación

In [ ]:
# Preparación del entorno: ejecuta esta celda la primera (Colab, VS Code o Codespaces)
import sys, os, subprocess, importlib.util
EN_COLAB = "google.colab" in sys.modules
print("Entorno:", "Google Colab" if EN_COLAB else "Jupyter en VS Code / Codespaces")
paquetes = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "matplotlib": "matplotlib",
            "seaborn": "seaborn", "joblib": "joblib", "xgboost": "xgboost", "lightgbm": "lightgbm"}
faltan = [pip for mod, pip in paquetes.items() if importlib.util.find_spec(mod) is None]
if faltan:
    print("Instalando:", ", ".join(faltan))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])

# Funciones del curso en ia_utils.py (no modifica tu utils.py)
IA_UTILS = "\"\"\"Funciones del curso IA aplicada a la ciberseguridad (sesiones 3 a 6).\"\"\"\nimport os, re, urllib.request\nimport numpy as np\nimport pandas as pd\n\nNSL_URL = \"https://raw.githubusercontent.com/defcom17/NSL_KDD/master/\"\nNSL_COLS = [\"duration\",\"protocol_type\",\"service\",\"flag\",\"src_bytes\",\"dst_bytes\",\"land\",\n  \"wrong_fragment\",\"urgent\",\"hot\",\"num_failed_logins\",\"logged_in\",\"num_compromised\",\n  \"root_shell\",\"su_attempted\",\"num_root\",\"num_file_creations\",\"num_shells\",\n  \"num_access_files\",\"num_outbound_cmds\",\"is_host_login\",\"is_guest_login\",\"count\",\n  \"srv_count\",\"serror_rate\",\"srv_serror_rate\",\"rerror_rate\",\"srv_rerror_rate\",\n  \"same_srv_rate\",\"diff_srv_rate\",\"srv_diff_host_rate\",\"dst_host_count\",\n  \"dst_host_srv_count\",\"dst_host_same_srv_rate\",\"dst_host_diff_srv_rate\",\n  \"dst_host_same_src_port_rate\",\"dst_host_srv_diff_host_rate\",\"dst_host_serror_rate\",\n  \"dst_host_srv_serror_rate\",\"dst_host_rerror_rate\",\"dst_host_srv_rerror_rate\",\n  \"label\",\"difficulty\"]\n\n\ndef load_nsl_kdd(split=\"train\", base=\"data\"):\n    \"\"\"Carga NSL-KDD y añade la columna is_attack. Descarga los ficheros si faltan.\"\"\"\n    name = \"KDDTrain+.txt\" if split == \"train\" else \"KDDTest+.txt\"\n    path = os.path.join(base, name)\n    if not os.path.exists(path):\n        os.makedirs(base, exist_ok=True)\n        urllib.request.urlretrieve(NSL_URL + name.replace(\"+\", \"%2B\"), path)\n    df = pd.read_csv(path, header=None, names=NSL_COLS)\n    df[\"is_attack\"] = (df[\"label\"] != \"normal\").astype(int)\n    return df\n\n\nATTACK_MAP = {\"normal\": \"normal\",\n  **{a: \"dos\" for a in [\"back\", \"land\", \"neptune\", \"pod\", \"smurf\", \"teardrop\",\n                        \"apache2\", \"udpstorm\", \"processtable\", \"mailbomb\"]},\n  **{a: \"probe\" for a in [\"satan\", \"ipsweep\", \"nmap\", \"portsweep\", \"mscan\", \"saint\"]},\n  **{a: \"r2l\" for a in [\"guess_passwd\", \"ftp_write\", \"imap\", \"phf\", \"multihop\",\n                        \"warezmaster\", \"warezclient\", \"spy\", \"xlock\", \"xsnoop\",\n                        \"snmpguess\", \"snmpgetattack\", \"httptunnel\", \"sendmail\",\n                        \"named\", \"worm\"]},\n  **{a: \"u2r\" for a in [\"buffer_overflow\", \"loadmodule\", \"rootkit\", \"perl\",\n                        \"sqlattack\", \"xterm\", \"ps\"]}}\n\n\ndef add_category(df):\n    \"\"\"Añade la columna category: normal, dos, probe, r2l o u2r.\"\"\"\n    df[\"category\"] = df[\"label\"].map(ATTACK_MAP)\n    return df\n\n\ndef generar_flujos(n=60_000, frac_ataque=0.25, seed=0, cambio_herramienta=False):\n    \"\"\"Flujos de red sintéticos con columnas al estilo CICIDS2017, incluidos sus defectos\n    (espacios en los nombres, infinitos, nulos y duplicados). Etiquetas BENIGN / DDoS.\n    Con cambio_herramienta=True el ataque llega en dos episodios con herramientas distintas.\"\"\"\n    rng = np.random.default_rng(seed)\n    nb, na = int(n * (1 - frac_ataque)), int(n * frac_ataque)\n    t0 = pd.Timestamp(\"2017-07-07 09:00:00\")\n    # --- tráfico benigno\n    port_b = rng.choice([443, 80, 53, 22, 8080, 3389, 0], nb, p=[.40, .20, .20, .05, .05, .02, .08])\n    port_b = np.where(port_b == 0, rng.integers(1024, 65535, nb), port_b)\n    dur_b = np.where(rng.random(nb) < 0.02, 0, rng.lognormal(11, 2, nb)).astype(int)\n    fwd_b = rng.poisson(8, nb) + 1\n    bwd_b = rng.poisson(7, nb)\n    fwl_b = np.where(rng.random(nb) < 0.15, rng.uniform(0, 10, nb), rng.normal(400, 250, nb)).clip(0)\n    bwl_b = rng.normal(700, 400, nb).clip(0)\n    syn_b = (rng.random(nb) < 0.3).astype(int)\n    win_b = rng.choice([29200, 65535, 8192, 14600, 64240, -1], nb, p=[.25, .25, .15, .1, .15, .1])\n    ts_b = rng.uniform(0, 8 * 3600, nb)\n    # --- ataque\n    if cambio_herramienta:\n        epi = rng.random(na) < 0.5\n        ts_a = np.where(epi, rng.uniform(1 * 3600, 2 * 3600, na), rng.uniform(5 * 3600, 6 * 3600, na))\n        port_a = np.where(epi, 80, rng.choice([8080, 443], na))\n        win_a = np.where(epi, 8192, rng.choice([29200, 65535, 64240], na))\n        fwl_a = np.where(epi, rng.normal(8, 6, na), rng.normal(60, 40, na)).clip(0)\n    else:\n        ts_a = rng.uniform(6.9 * 3600, 7.3 * 3600, na)\n        port_a = np.where(rng.random(na) < 0.9, 80, 443)\n        win_a = np.where(rng.random(na) < 0.95, 8192, 29200)\n        fwl_a = rng.normal(8, 6, na).clip(0)\n    dur_a = rng.lognormal(13, 1.5, na).astype(int)\n    fwd_a = rng.poisson(4, na) + 1\n    bwd_a = rng.poisson(3, na)\n    bwl_a = rng.normal(1500, 600, na).clip(0)\n    syn_a = (rng.random(na) < 0.6).astype(int)\n\n    def bloque(port, dur, fwd, bwd, fwl, bwl, syn, win, ts, label):\n        tot_f, tot_b = fwd * fwl, bwd * bwl\n        seg = np.maximum(dur, 0) / 1e6\n        with np.errstate(divide=\"ignore\", invalid=\"ignore\"):\n            fbs = (tot_f + tot_b) / seg\n            fps = (fwd + bwd) / seg\n            iat = np.where(fwd + bwd > 1, dur / np.maximum(fwd + bwd - 1, 1), np.nan)\n        return pd.DataFrame({\n            \" Destination Port\": port, \" Flow Duration\": dur, \" Total Fwd Packets\": fwd,\n            \" Total Backward Packets\": bwd, \"Total Length of Fwd Packets\": tot_f.round(1),\n            \" Total Length of Bwd Packets\": tot_b.round(1), \" Fwd Packet Length Mean\": fwl.round(2),\n            \" Bwd Packet Length Mean\": bwl.round(2), \"Flow Bytes/s\": fbs, \" Flow Packets/s\": fps,\n            \" Flow IAT Mean\": iat, \" SYN Flag Count\": syn, \" ACK Flag Count\": (rng.random(len(port)) < 0.85).astype(int),\n            \"Init_Win_bytes_forward\": win, \" Avg Fwd Segment Size\": fwl.round(2),\n            \" Timestamp\": t0 + pd.to_timedelta(ts, unit=\"s\"), \" Label\": label})\n\n    df = pd.concat([bloque(port_b, dur_b, fwd_b, bwd_b, fwl_b, bwl_b, syn_b, win_b, ts_b, \"BENIGN\"),\n                    bloque(port_a, dur_a, fwd_a, bwd_a, fwl_a, bwl_a, syn_a, win_a, ts_a, \"DDoS\")], ignore_index=True)\n    df.loc[rng.random(len(df)) < 0.001, \"Flow Bytes/s\"] = np.nan\n    df = pd.concat([df, df.sample(frac=0.01, random_state=seed)], ignore_index=True)\n    return df.sort_values(\" Timestamp\").reset_index(drop=True)\n\n\ndef generar_authlog(ruta=\"auth.log\", n_atacantes=12, seed=42):\n    \"\"\"auth.log sintético de la sesión 2: 40 IP legítimas y n_atacantes IP de fuerza bruta (203.0.113.0/24).\"\"\"\n    import datetime as dt\n    rng = np.random.default_rng(seed); start = dt.datetime(2026, 10, 1)\n    users = [\"root\", \"admin\", \"carlos\", \"ana\", \"luis\", \"deploy\", \"test\", \"oracle\", \"ubuntu\", \"postgres\"]\n    rows = []\n    for i in range(40):\n        ip = f\"10.10.10.{10 + i}\"\n        for t in rng.uniform(0, 7 * 24 * 3600, rng.integers(5, 40)):\n            rows.append((start + dt.timedelta(seconds=float(t)), bool(rng.random() > 0.08), str(rng.choice(users[2:6])), ip))\n    for i in range(n_atacantes):\n        ip = f\"203.0.113.{20 + i}\"; t0 = rng.uniform(0, 7 * 24 * 3600 - 3600)\n        for t in t0 + np.sort(rng.uniform(0, 1800, rng.integers(60, 400))):\n            rows.append((start + dt.timedelta(seconds=float(t)), False, str(rng.choice(users)), ip))\n    rows.sort(key=lambda r: r[0])\n    with open(ruta, \"w\") as f:\n        for ts, ok, user, ip in rows:\n            v = \"Accepted\" if ok else \"Failed\"\n            f.write(f\"{ts:%b %d %H:%M:%S} srv sshd[{rng.integers(1000, 9999)}]: {v} password \"\n                    f\"for {user} from {ip} port {rng.integers(1024, 65000)} ssh2\\n\")\n    return ruta\n\n\ndef auth_features(ruta=\"auth.log\"):\n    \"\"\"Características por IP y ventana de 10 minutos. La columna verdad solo sirve para evaluar.\"\"\"\n    pat = re.compile(r\"^(\\w{3}\\s+\\d+ \\d\\d:\\d\\d:\\d\\d) \\S+ sshd\\[\\d+\\]: (Failed|Accepted) password for (\\S+) from (\\S+) port\")\n    reg = []\n    for linea in open(ruta):\n        m = pat.match(linea)\n        if m:\n            reg.append((pd.to_datetime(\"2026 \" + m.group(1), format=\"%Y %b %d %H:%M:%S\"),\n                        m.group(2) == \"Accepted\", m.group(3), m.group(4)))\n    log = pd.DataFrame(reg, columns=[\"ts\", \"ok\", \"user\", \"ip\"])\n    log[\"win\"] = log[\"ts\"].dt.floor(\"10min\")\n    f = log.groupby([\"ip\", \"win\"]).agg(n=(\"ok\", \"size\"), n_ok=(\"ok\", \"sum\"), n_users=(\"user\", \"nunique\")).reset_index()\n    f[\"n_fail\"] = f[\"n\"] - f[\"n_ok\"]\n    f[\"fail_ratio\"] = f[\"n_fail\"] / f[\"n\"]\n    f[\"verdad\"] = f[\"ip\"].str.startswith(\"203.0.113.\").astype(int)\n    return f\n"
with open("ia_utils.py", "w", encoding="utf-8") as f:
    f.write(IA_UTILS)

import importlib, ia_utils; importlib.reload(ia_utils)
from ia_utils import *
tr, te = load_nsl_kdd("train"), load_nsl_kdd("test")      # descarga NSL-KDD si falta
import numpy as np, pandas as pd, matplotlib.pyplot as plt, sklearn
plt.rcParams["figure.figsize"] = (9, 4)
print("NSL-KDD:", tr.shape, te.shape, "| scikit-learn", sklearn.__version__)

## Parte A · Regresión lineal: lo esperado y lo anómalo
120 días de intentos fallidos en una VPN: tendencia, caída en fin de semana y **un ataque inyectado el día 100**. Si el modelo aprende lo normal, el ataque aparece como un **residuo enorme**.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
rng = np.random.default_rng(1)
dias = np.arange(120); finde = (dias % 7 >= 5).astype(int)
y = 200 + 1.5 * dias - 60 * finde + rng.normal(0, 15, 120)
y[100] += 400                                      # ataque de fuerza bruta
X = np.c_[dias, finde]
Xb = np.c_[np.ones(len(X)), X]                     # ecuación normal, a mano
w = np.linalg.solve(Xb.T @ Xb, Xb.T @ y)
print("pesos (b, día, finde):", w.round(2))
Xtr_, Xte_, ytr_, yte_ = X[:90], X[90:], y[:90], y[90:]      # división temporal
lr = LinearRegression().fit(Xtr_, ytr_); pred = lr.predict(Xte_)
print(f"RMSE {mean_squared_error(yte_, pred) ** 0.5:.1f} | MAE {mean_absolute_error(yte_, pred):.1f} | R2 {r2_score(yte_, pred):.2f}")

In [ ]:
res_tr = ytr_ - lr.predict(Xtr_); res = yte_ - pred
umbral = 3 * res_tr.std()
anom = np.abs(res) > umbral
print("días marcados:", dias[90:][anom])
plt.plot(dias, y, label="real"); plt.plot(dias[90:], pred, label="esperado")
plt.scatter(dias[90:][anom], yte_[anom], color="red", s=80, label="anomalía", zorder=3)
plt.legend(); plt.title("Residuo > 3 desviaciones típicas = alerta"); plt.show()

**Qué deberías ver:** pesos cercanos a 1,5 por día y −60 en fin de semana, y el día 100 marcado en rojo. Es exactamente lo que hace un umbral de alerta en un SIEM.

**Reto avanzado.** Descenso de gradiente: estandariza `X`, inicializa `w = 0` y aplica `w -= eta * 2 * Xs.T @ (Xs @ w + b - y) / n` durante 500 iteraciones. Prueba `eta` en 0,01, 0,1 y 1,1: ¿cuál converge, cuál es lenta y cuál diverge? Después ajusta `PolynomialFeatures(degree=15)` + `Ridge(alpha)` con alpha 0,01, 1 y 100 y compara RMSE de train y test.

In [ ]:
# Tu código

## Parte B · Regresión logística desde cero
La puntuación `z = w·x + b` pasa por la **sigmoide** para convertirse en probabilidad. Entrenar es bajar el **log-loss** paso a paso.

In [ ]:
f2 = ["serror_rate", "same_srv_rate"]
mu, sd = tr[f2].mean(), tr[f2].std()
A = ((tr[f2] - mu) / sd).to_numpy(); ya = tr.is_attack.to_numpy()
def sigmoid(z): return 1 / (1 + np.exp(-z))
def fit_logreg(X, y, eta=0.5, epochs=400):
    w, b, hist = np.zeros(X.shape[1]), 0.0, []
    for e in range(epochs):
        p = sigmoid(X @ w + b)
        w -= eta * X.T @ (p - y) / len(y); b -= eta * (p - y).mean()
        hist.append(-np.mean(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12)))
    return w, b, hist
w, b, hist = fit_logreg(A, ya)
from sklearn.linear_model import LogisticRegression
sk = LogisticRegression(C=1e6).fit(A, ya)
print("propio:", w.round(3), round(b, 3), "| scikit-learn:", sk.coef_[0].round(3), sk.intercept_.round(3))
plt.plot(hist); plt.title("log-loss por iteración"); plt.show()

In [ ]:
g1, g2 = np.meshgrid(np.linspace(-1, 2.5, 200), np.linspace(-2.5, 1, 200))
Z = sigmoid(np.c_[g1.ravel(), g2.ravel()] @ w + b).reshape(g1.shape)
plt.contourf(g1, g2, Z, levels=20, alpha=0.5, cmap="RdBu_r")
plt.scatter(A[::60, 0], A[::60, 1], c=ya[::60], s=6, cmap="coolwarm")
plt.xlabel("serror_rate (estandarizada)"); plt.ylabel("same_srv_rate (estandarizada)"); plt.title("Frontera de decisión: una recta"); plt.show()

**Qué deberías ver:** el log-loss baja y se estabiliza, pesos parecidos a los de scikit-learn y una frontera recta entre la zona roja y la azul.

**Reto avanzado.** Cambia `eta` a 0,01 y a 20 y describe convergencia y divergencia. ¿Por qué se vigila el log-loss y no solo la exactitud? ¿Por qué la frontera es necesariamente una recta?

In [ ]:
# Tu código

## Parte C · NSL-KDD completo: odds y umbral por coste

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.metrics import precision_score, recall_score, confusion_matrix
num = ["duration", "src_bytes", "dst_bytes", "count", "srv_count", "serror_rate", "rerror_rate", "same_srv_rate",
       "diff_srv_rate", "dst_host_count", "dst_host_srv_count", "dst_host_same_srv_rate", "logged_in", "hot", "num_failed_logins"]
cat = ["protocol_type", "flag"]
prep = ColumnTransformer([("n", make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()), num),
                          ("c", OneHotEncoder(handle_unknown="ignore"), cat)])
clf = make_pipeline(prep, LogisticRegression(C=1.0, max_iter=2000)).fit(tr[num + cat], tr.is_attack)
coef = pd.Series(clf[-1].coef_[0], index=clf[0].get_feature_names_out()).sort_values()
pd.concat([coef.head(5), coef.tail(5)]).plot.barh(title="Lo que más empuja hacia normal (−) y hacia ataque (+)"); plt.show()
print("odds ratio de serror_rate:", round(np.exp(coef["n__serror_rate"]), 2))

In [ ]:
prob = clf.predict_proba(te[num + cat])[:, 1]
filas = []
for t in [0.05, 0.1, 0.3, 0.5, 0.7, 0.9]:
    pred = (prob > t).astype(int)
    tn, fp, fn, tp = confusion_matrix(te.is_attack, pred).ravel()
    filas.append({"umbral": t, "recall": tp / (tp + fn), "precision": tp / (tp + fp), "FN": fn, "FP": fp, "coste (50·FN + FP)": 50 * fn + fp})
tabla = pd.DataFrame(filas).round(3); print(tabla)
print("umbral de coste mínimo:", tabla.loc[tabla["coste (50·FN + FP)"].idxmin(), "umbral"])

**Qué deberías ver:** con un falso negativo 50 veces más caro, el umbral de coste mínimo está muy por debajo de 0,5. **El umbral es una decisión de negocio**, y en un proyecto real se elige en validación, no en test.

**Reto avanzado.** Repite con `class_weight="balanced"` y compara la tabla. Elige el umbral con una partición de validación separada del train y aplícalo una sola vez al test.

In [ ]:
# Tu código

## Parte D · Naive Bayes para texto de phishing
Conjunto de juguete: lo importante es ver a Bayes funcionando. El detector de phishing real llega en la sesión 13.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
textos = ["urgente verifique su cuenta bancaria ahora", "su paquete esta retenido pague la tasa", "has ganado un premio haz clic aqui",
          "su cuenta sera suspendida confirme su contrasena", "factura pendiente descargue el adjunto urgente", "alerta de seguridad inicie sesion ya",
          "reunion de equipo manana a las 10", "adjunto el informe trimestral para revision", "recordatorio de la formacion de ciberseguridad",
          "gracias por tu ayuda con el despliegue", "acta de la reunion del comite", "planificacion de vacaciones del departamento"]
y_txt = [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
vec = CountVectorizer(); Xt = vec.fit_transform(textos)
nb = MultinomialNB(alpha=1.0).fit(Xt, y_txt)
nuevos = ["urgente confirme su cuenta", "adjunto el acta de la reunion", "pague ahora su factura"]
for t, pr in zip(nuevos, nb.predict_proba(vec.transform(nuevos))[:, 1]):
    print(f"{pr:.3f}  {t}")
palabras = np.array(vec.get_feature_names_out())
print("palabras más phishing:", palabras[np.argsort(nb.feature_log_prob_[1] - nb.feature_log_prob_[0])[-5:]])

**Reto avanzado.** Calcula a mano P(phishing | «urgente») con las frecuencias de este conjunto. Explica qué hace `alpha` (suavizado de Laplace) y añade tres mensajes tuyos que engañen al modelo.

In [ ]:
# Tu código

## Parte E · k-NN y SVM: el efecto del escalado

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
import time
from sklearn.model_selection import train_test_split
muestra = tr.sample(12000, random_state=0)       # validación dentro de train: misma distribución
sub, tes = train_test_split(muestra, test_size=4000, stratify=muestra.is_attack, random_state=0)
res = []
for nombre, m in [("k-NN sin escalar", KNeighborsClassifier(5)),
                  ("k-NN escalado", make_pipeline(StandardScaler(), KNeighborsClassifier(5))),
                  ("k-NN log1p + escalado", make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), KNeighborsClassifier(5))),
                  ("SVM lineal", make_pipeline(StandardScaler(), LinearSVC())),
                  ("SVM RBF", make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale")))]:
    t0 = time.perf_counter(); m.fit(sub[num], sub.is_attack); t1 = time.perf_counter(); pr = m.predict(tes[num]); t2 = time.perf_counter()
    res.append({"modelo": nombre, "acc_test": (pr == tes.is_attack).mean(), "t_fit_s": t1 - t0, "t_pred_s": t2 - t1})
print(pd.DataFrame(res).round(3))

**Qué deberías ver:** una sorpresa útil. Sin escalar, la distancia la decide casi solo `src_bytes`, que en NSL-KDD es muy informativa, y el escalado solo no mejora. Con **log1p + escalado** (colas pesadas domadas) el k-NN llega a 0,99. El SVM RBF tarda bastante más que el lineal. **Lección de método: el escalado lo exige el algoritmo, pero qué preprocesado gana se mide, no se supone.**

**Reto avanzado.** Calcula qué fracción de la distancia media entre dos flujos aporta `src_bytes` sin escalar y escalada. Duplica el tamaño de `sub` y mide cómo cambian `t_fit_s` y `t_pred_s` de cada modelo. ¿Cuándo compensa el kernel RBF frente al lineal en un IDS con millones de flujos?

In [ ]:
# Tu código

## Parte F · Clasificación multiclase de ataques

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
tr, te = add_category(tr), add_category(te)
multi = make_pipeline(prep, LogisticRegression(max_iter=3000, class_weight="balanced")).fit(tr[num + cat], tr["category"])
pm = multi.predict(te[num + cat])
print(classification_report(te["category"], pm, zero_division=0))
ConfusionMatrixDisplay.from_predictions(te["category"], pm, xticks_rotation=45); plt.show()

**Qué deberías ver:** `dos` y `probe` se detectan bien; `r2l` y `u2r` tienen el peor recall: pocos ejemplos, comportamiento parecido al normal y ataques del test que no estaban en train.

## Entrega
Reinicia y ejecuta todo. Entrega `ia4_modelos.ipynb` con la tabla de umbral por coste y tu conclusión: ¿es 0,5 el umbral óptimo para un SOC?